## Step 1: Load and Split the Iris Dataset


In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

iris = load_iris()

X = pd.DataFrame(data=iris['data'], columns=iris.feature_names)
y = pd.DataFrame(data=iris['target'], columns=['target'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=43
)


## Step 2: Implement train(X_train, y_train)


In [2]:
def train(X_train, y_train):
    y_1d = np.asarray(y_train).ravel()

    # Binary targets: 1 if sample belongs to that class, else 0
    y0 = (y_1d == 0).astype(int)
    y1 = (y_1d == 1).astype(int)
    y2 = (y_1d == 2).astype(int)

    clf0 = LogisticRegression(max_iter=1000).fit(X_train, y0)
    clf1 = LogisticRegression(max_iter=1000).fit(X_train, y1)
    clf2 = LogisticRegression(max_iter=1000).fit(X_train, y2)

    return clf0, clf1, clf2

clf0, clf1, clf2 = train(X_train, y_train)


## Step 3: Implement predict_one_vs_all(X, clf0, clf1, clf2)


In [ ]:
def predict_one_vs_all(X, clf0, clf1, clf2):
    prob0 = clf0.predict_proba(X)[:, 1]
    prob1 = clf1.predict_proba(X)[:, 1]
    prob2 = clf2.predict_proba(X)[:, 1]

    # Stack probabilities into shape  and choose highest probability
    all_probs = np.column_stack([prob0, prob1, prob2])
    classes = np.argmax(all_probs, axis=1)

    return classes

custom_preds = predict_one_vs_all(X_test, clf0, clf1, clf2)
print("Custom One-vs-Rest predictions:\n", custom_preds)


Custom One-vs-Rest predictions:
 [0 0 2 1 2 0 2 1 1 1 0 1 2 0 1 1 0 0 2 2 0 0 0 2 2 2 0 1 0 0]



## Step 4: Compare with Scikit-learn's One-vs-Rest Logistic Regression


In [ ]:
# Scikit-learn One-vs-Rest wrapper
ovr_sklearn = OneVsRestClassifier(LogisticRegression(max_iter=1000))
ovr_sklearn.fit(X_train, y_train.values.ravel())
sklearn_ovr_preds = ovr_sklearn.predict(X_test)

# Standard Scikit-learn LogisticRegression 
multinomial_sklearn = LogisticRegression(max_iter=1000)
multinomial_sklearn.fit(X_train, y_train.values.ravel())
sklearn_multi_preds = multinomial_sklearn.predict(X_test)

y_test_1d = y_test.values.ravel()

print("Custom OvR vs Scikit-learn OvR exact match:", np.array_equal(custom_preds, sklearn_ovr_preds))
print(f"Custom OvR Test Accuracy:          {np.mean(custom_preds == y_test_1d) * 100:.2f}%")
print(f"Scikit-learn OvR Test Accuracy:    {np.mean(sklearn_ovr_preds == y_test_1d) * 100:.2f}%")
print(f"Scikit-learn Multi Test Accuracy:  {np.mean(sklearn_multi_preds == y_test_1d) * 100:.2f}%")


Custom OvR vs Scikit-learn OvR exact match: True
Custom OvR Test Accuracy:          100.00%
Scikit-learn OvR Test Accuracy:    100.00%
Scikit-learn Multi Test Accuracy:  100.00%
